In [ ]:
! pwd

/home/sabrina/Documents/BioHack26


In [ ]:
! python samap_3species/scripts/03_build_mmseqs_maps.py

Working directory: /home/sabrina/Documents/BioHack26/samap_3species/tmp/mmseqs_mm_dr_14p1v8s7
createdb /home/sabrina/Documents/BioHack26/samap_3species/refs/samap/mouse.GRCm38.GENCODEvM12.samap.fa /home/sabrina/Documents/BioHack26/samap_3species/tmp/mmseqs_mm_dr_14p1v8s7/mm --dbtype 2 

MMseqs Version:                    	18.8cc5c
Database type                      	2
Shuffle input database             	true
Createdb mode                      	0
Write lookup file                  	1
Offset of numeric ids              	0
Threads                            	32
Compressed                         	0
Mask residues                      	0
Mask residues probability          	0.9
Mask lower case residues           	0
Mask lower letter repeating N times	0
Use GPU                            	0
Verbosity                          	3

Converting sequences
[45451] 0s 55ms
Time for merging to mm_h: 0h 0m 0s 3ms
Time for merging to mm: 0h 0m 0s 46ms
Database type: Nucleotide
Time for processing: 0h 0m

In [ ]:
from samap import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder,
                            sankey_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samap.sam import SAM
import pandas as pd

import scanpy as sc
import anndata as ad
import numpy as np
from pathlib import Path
import scipy.sparse as sp
from samap.utils import save_samap, load_samap
import matplotlib.colors as mcolors

In [ ]:
RAW_DIR = Path('/media/sabrina/hdd1/BioHack26')
PROJECT_DIR = Path('/home/sabrina/Documents/BioHack26/samap_3species')
PROCESSED_DIR = PROJECT_DIR / "data" / "processed_h5ad"
SAMAP_OUTPUT_DIR = PROJECT_DIR / "data" / "samap"
MAPS_DIR = PROJECT_DIR / "maps_mmseqs" 

RAW_FILES = {
    "human": RAW_DIR / "221024LimbCellranger3annotated_filtered_adjusted_skm_lineage_raw_counts_20260817.h5ad",
    "mouse": RAW_DIR / "MusMusculus_atlas_musclecells.h5ad",
    "zebrafish": RAW_DIR / "Zebrafish_dev_atlas.h5ad",
}
HUMAN_READY_PATH = PROCESSED_DIR / "human_SKM_samap_ready.h5ad"
MOUSE_READY_PATH = PROCESSED_DIR / "mouse_SKM_samap_ready.h5ad"
ZEBRAFISH_READY_PATH = PROCESSED_DIR / "zebrafish_SKM_samap_ready.h5ad"
MOUSE_ANNOTATED_PATH = PROCESSED_DIR / "mouse_annot_SKM_samap_ready.h5ad"
SAMAP_H5AD_PATH = Path("samap_mm_dr_annot.h5ad")
SAMAP_OBJECT_PATH = SAMAP_OUTPUT_DIR / "SamapObject_mm_dr_annot"

# Default: reuse the saved manual annotations, clustering, and embeddings.
# Rebuilding requires the complete, verified cluster-to-cell-type mapping below.
REBUILD_MOUSE_ANNOTATION = False  # False
MANUAL_ANNOTATIONS = {}  # {}# Fill only with verified labels for every leiden_re_0.4 cluster.

In [ ]:
fn1 = str(MOUSE_ANNOTATED_PATH)
fn2 = str(ZEBRAFISH_READY_PATH)

In [ ]:
filenames = {'mm':fn1, 'dr':fn2}

In [ ]:
keys = {'mm':'cell_type', 'dr':'cell_type'} #'hs':'celltype',

In [ ]:
for sid, path in filenames.items():
    print("\n", sid, "→", path)
    input_adata = sc.read_h5ad(path)
    print("  shape:", input_adata.shape)
    print("  n_vars:", input_adata.n_vars)
    print("  len(var_names):", len(input_adata.var_names))


 mm → /home/sabrina/Documents/BioHack26/samap_3species/data/processed_h5ad/mouse_annot_SKM_samap_ready.h5ad
  shape: (305003, 45525)
  n_vars: 45525
  len(var_names): 45525

 dr → /home/sabrina/Documents/BioHack26/samap_3species/data/processed_h5ad/zebrafish_SKM_samap_ready.h5ad
  shape: (101871, 32031)
  n_vars: 32031
  len(var_names): 32031


In [ ]:
sm = SAMAP(filenames, f_maps=str(MAPS_DIR) + "/")

INFO: Using backend: cpu
INFO: 28679 'mm' gene symbols match between the datasets and the BLAST graph.
INFO: 23383 'dr' gene symbols match between the datasets and the BLAST graph.


In [ ]:
sm.run()
samap = sm.samap

INFO: Prepping datasets for translation.
INFO: Translating feature spaces pairwise.
INFO: Projecting data into joint latent space. 53.80s
INFO: Correcting data with means. 0.47s
INFO: Expanding neighbourhoods of species mm...
INFO: Expanding neighbourhoods of species dr...
INFO: Indegree coarsening
INFO: Rescaling edge weights by expression correlations.


In [ ]:
D, MappingTable = get_mapping_scores(sm, keys, n_top=0)

In [ ]:
# Save mapping tables using the original delimiter and index settings.
SAMAP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df = pd.DataFrame(D)
df1 = pd.DataFrame(MappingTable)
df.to_csv(SAMAP_OUTPUT_DIR / "MappingD_mm_dr_annot_map2.csv", sep="\t", index=False)
df1.to_csv(SAMAP_OUTPUT_DIR / "MappingT_mm_dr_annot_map2.csv", sep="\t", index=False)

In [ ]:
# Interactive sankey plot

import numpy as np
import pandas as pd
import holoviews as hv
hv.extension('bokeh')  # 'matplotlib' if svg saving
hv.output(size=100)


align_thr = 0.1

# Keep mouse rows ('mm_') and zebrafish columns ('dr_').
mapping_filtered = MappingTable.loc[
    [ix for ix in MappingTable.index if ix.startswith("mm_")],
    [col for col in MappingTable.columns if col.startswith("dr_")]
]

# threshold
d = mapping_filtered.values.copy()
d[d < align_thr] = 0

# sankey preparation
x, y = d.nonzero()
values = d[x, y]
source_labels = mapping_filtered.index[x]
target_labels = mapping_filtered.columns[y]

# Sankey dataframe 
relations_df = pd.DataFrame({
    "source": source_labels,
    "target": target_labels,
    "Value": values
})

# Sankey plot with orientation mm → dr
sankey = hv.Sankey(relations_df, kdims=["source", "target"], vdims=["Value"]).opts(
    show_values=False,
    node_padding=10,
    width=900,
    height=600,
    label_position='left',
    node_sort=True
)

# print
sankey

In [ ]:
from scipy import sparse
for species, sam in sm.sams.items():
    X = sam.adata.X

    if not sparse.issparse(X):
        print(f"{species} : dense matrix; conversion skipped")
        continue

    X = X.tocsr(copy=False)

    print(
        f"{species} : shape={X.shape}, nnz={X.nnz:,}, "
        f"indices={X.indices.dtype}, indptr={X.indptr.dtype}"
    )

    # Use 64-bit column indices and row pointers
    X.indices = X.indices.astype(np.int64, copy=False)
    X.indptr = X.indptr.astype(np.int64, copy=False)

    sam.adata.X = X

In [ ]:
# Gene pairs analysis

gpf = GenePairFinder(sm,keys=keys)

gene_pairs = gpf.find_all(align_thr=0.1)

df = pd.DataFrame(gene_pairs)

df.to_csv('GenePair_annot_map2.tsv', sep='\t', index=False) 

In [ ]:
sc.pl.umap(samap.adata, color=['mm_cell_type', 'dr_cell_type', 'species'], frameon=False, title='') #, save='umap1.svg'

In [ ]:
sm.samap.adata.write(Path("samap_mm_dr_annot_map2.h5ad"))

In [ ]:
save_samap(sm, str(SAMAP_OUTPUT_DIR / "SamapObject_mm_dr_annot_map2"))